# D3b 토큰화·임베딩·텍스트 분류 — 실습 (W9, D3 2부작 완결)

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.
> (20 Newsgroups 데이터가 처음 실행 시 자동 다운로드됩니다.)

**이 실습이 끝나면**
1. 미니 코퍼스로 **토큰화→사전→정수 인코딩**을 손으로 놓는다
2. **임베딩 = 행 꺼내기** (`emb(i) == emb.weight[i]`)를 검산한다
3. 진짜 뉴스를 전처리한다 — 상위 5,000 사전(**train만!**) + **앞쪽 패딩**
4. LSTM 분류기(파라미터 353,538 — 임베딩 90%)를 학습한다 — **~74%**
5. 틀린 문서와 **내 문장**으로 모델을 진단한다

**7단계 멘탈모델 초점:** 표현(주인공!) + 모델

## Part A. 미니 코퍼스 — 토큰화·사전·정수 인코딩 손으로 ⭐
예약석: `<pad>`=0(빈칸), `<unk>`=1(모르는 단어).

In [ ]:
import re                                               # 정규식(토큰화)
import torch                                            # PyTorch
import torch.nn as nn                                   # 신경망 모듈
import matplotlib.pyplot as plt                         # 그래프

torch.manual_seed(0)                                    # 재현성
def tok(t):                                             # ① 토큰화: 소문자 영단어만
    return re.findall(r'[a-z]+', t.lower())

sents = ['i love space', 'space is far', 'i love baseball']  # 미니 코퍼스 3문장
vocab = {'<pad>': 0, '<unk>': 1}                        # ② 사전 — 예약석 2개 먼저!
for s in sents:
    for w in tok(s):
        vocab.setdefault(w, len(vocab))                 # 새 단어면 다음 번호 부여
print('사전:', vocab)                                   # i=2, love=3, space=4, ...

def encode_mini(s):                                     # 문장 → 정수 시퀀스
    return [vocab.get(w, ___) for w in tok(s)]          # ✍️ 빈칸: 사전에 없으면? (<unk>의 번호)
print('"i love space"  →', encode_mini('i love space'))     # [2, 3, 4]
print('"i love pytorch" →', encode_mini('i love pytorch'))  # [2, 3, 1] — pytorch는 <unk>!

## Part B. 임베딩 = 표에서 행 꺼내기 검산 ⭐
`nn.Embedding(V, E)` = V행 E열 행렬. i번 단어의 벡터 = **weight의 i번째 행** — 직접 확인합니다.

In [ ]:
emb_demo = nn.Embedding(len(vocab), 4)                  # ③ 임베딩: 8단어 × 4차원 표
print('표(weight) shape:', emb_demo.weight.shape)       # (8, 4) — 그냥 행렬!

idx = vocab['space']                                    # space의 번호(4)
lookup = emb_demo(torch.tensor(idx))                    # 임베딩 통과 = ?
row = emb_demo.weight[___]                              # ✍️ 빈칸: 표의 몇 번째 행과 같을까
print('emb(space) == weight의 그 행?', torch.allclose(lookup, row))  # True — 행 꺼내기가 전부
print('space 벡터:', lookup.detach().round(decimals=2)) # 지금은 난수 — 학습되면 의미를 얻음

> 표의 값들은 **학습되는 파라미터**(D1의 W와 같은 신분). 분류를 잘하도록 갱신되다 보면 비슷하게 쓰이는 단어끼리 비슷한 벡터가 됩니다.

## Part C. 진짜 뉴스 전처리 — 사전은 train으로만! (M2)
야구 vs 우주. 상위 5,000 단어 사전 + MAX=200 **앞쪽 패딩**.

In [ ]:
from sklearn.datasets import fetch_20newsgroups          # 뉴스 데이터(첫 실행 시 다운로드)
from collections import Counter                          # 단어 빈도

cats = ['rec.sport.baseball', 'sci.space']               # 두 주제
tr = fetch_20newsgroups(subset='train', categories=cats, remove=('headers','footers','quotes'))
te = fetch_20newsgroups(subset='test',  categories=cats, remove=('headers','footers','quotes'))

counter = Counter(w for d in tr.data for w in tok(d))    # ⚠️ 빈도는 train으로만(M2 누수 방지)
itos = ['<pad>', '<unk>'] + [w for w, _ in counter.most_common(5000)]  # 예약 2 + 상위 5000
stoi = {w: i for i, w in enumerate(itos)}                # 단어→정수

MAX = 200                                                # ④ 패딩: 고정 길이
def encode(t):                                           # 문서 → 길이 200 정수 시퀀스
    ids = [stoi.get(w, 1) for w in tok(t)][:MAX]         # 인코딩(+길면 자르기)
    return [0] * (___ - len(ids)) + ids                  # ✍️ 빈칸: 앞쪽을 <pad>로 채워 길이 맞춤

Xtr = torch.tensor([encode(d) for d in tr.data]); ytr = torch.tensor(tr.target)
Xte = torch.tensor([encode(d) for d in te.data]); yte = torch.tensor(te.target)
print('train:', tuple(Xtr.shape), '| test:', tuple(Xte.shape))  # (~1190, 200) / (~790, 200)
print('사전 크기:', len(itos), '| 클래스:', list(tr.target_names))

> **앞쪽 패딩인 이유:** 분류는 **마지막 은닉**을 쓰므로(D3a), 마지막이 pad가 아니라 **실제 내용 직후**여야 신선한 요약이 됩니다.

## Part D. LSTM 분류기 조립 — 파라미터는 어디에?
Embedding → LSTM → h[-1](문서 요약) → Linear.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed=64, hidden=64, nclass=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed, padding_idx=___)  # ✍️ 빈칸: pad 벡터 0 고정
        self.lstm = nn.LSTM(embed, hidden, batch_first=___)  # ✍️ 빈칸: (배치,시간,특징) 규격(D3a!)
        self.fc = nn.Linear(hidden, nclass)              # 요약 → 클래스 점수
    def forward(self, x):
        e = self.embedding(x)                            # (B, 200) → (B, 200, 64)
        out, (h, c) = self.lstm(e)                       # 읽기 (LSTM 반환: 벨트 c 포함 — D3a)
        return self.fc(h[___])                           # ✍️ 빈칸: 마지막 층의 은닉(음수 인덱싱)

model = LSTMClassifier(len(itos))                        # 생성
n_total = sum(p.numel() for p in model.parameters())     # 전체(D1c numel)
n_emb = sum(p.numel() for p in model.embedding.parameters())  # 임베딩만
print('전체:', f'{n_total:,}', '| 임베딩:', f'{n_emb:,}',
      f'({100 * n_emb / n_total:.1f}%)')                 # 353,538 중 320,128 = 90.5% — 사전이 무겁다!

## Part E. 학습 & 평가 — 루프는 D1c 그대로

In [ ]:
from torch.utils.data import TensorDataset, DataLoader   # 배치 공급

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=32, shuffle=True)
criterion = nn.CrossEntropyLoss()                        # 분류 손실(D1b)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # Adam

losses = []                                              # epoch 손실 기록
for epoch in range(5):                                   # 5바퀴
    model.train(); running = 0.0
    for xb, yb in train_loader:                          # D1c 5단계 그대로
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        running += loss.item()
    losses.append(running / len(train_loader))
    print(f'epoch {epoch+1}: train loss = {losses[-1]:.4f}')

model.eval()                                             # 평가(D1c 스위치)
with torch.no_grad():
    acc = (model(Xte).argmax(___) == yte).float().mean().item()  # ✍️ 빈칸: 클래스 차원
print('test accuracy:', round(acc, 3))                   # ~0.74 (기준선 0.5)

plt.figure(figsize=(6, 3.5))                             # 손실 곡선
plt.plot(range(1, 6), losses, 'o-')                      # epoch별
plt.xlabel('epoch'); plt.ylabel('train loss')            # 축(영어)
plt.xticks(range(1, 6))                                  # 정수 눈금
plt.title('Text classifier training (baseball vs space)')  # 제목(영어)
plt.grid(True); plt.show()                               # 꾸준한 감소 확인

## Part F. 진단 — 틀린 문서와 내 문장
모델을 눈으로 만져 보는 가장 빠른 방법.

In [ ]:
with torch.no_grad():
    pred_all = model(Xte).argmax(1)                      # 전체 시험 예측
wrong = (pred_all != yte).nonzero().squeeze()            # 틀린 문서 인덱스
print(f'틀린 문서: {len(wrong)}개 / {len(yte)}개')       # ~23%
i = wrong[0].item()                                      # 첫 틀린 문서 들여다보기
print(f'실제={te.target_names[yte[i]]} | 예측={te.target_names[pred_all[i]]}')
print('본문 앞부분:', ' '.join(tok(te.data[i])[:40]), '...')  # 왜 틀렸을지 직접 판단

my_sents = ['the rocket will launch into orbit tomorrow',      # 내 문장 테스트
            'he hit a home run in the ninth inning',
            'the pitcher threw the ball to the moon']           # 일부러 섞은 문장!
with torch.no_grad():
    for s in my_sents:
        p = model(torch.tensor([encode(s)])).argmax(1).item()  # 같은 파이프라인으로 인코딩
        print(f'"{s}" → {te.target_names[p]}')           # 모델의 판단은?

> 세 번째 문장(투수가 달로 공을 던졌다)은 **야구·우주 단서가 섞인 함정** — 모델이 어느 쪽 단서에 더 끌리는지 관찰하세요. 짧은 임의 문장은 학습 분포(뉴스 본문)와 달라 더 자주 틀립니다.

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "`emb(i) == emb.weight[i]`가 왜 성립하는지 내가 설명할 테니 채점해 줘."
- "사전을 train으로만 만드는 이유를 M2 누수로 설명해 볼게."
- "내 문장 3개의 예측 결과를 보여줄 테니, 모델이 어떤 단어에 끌렸을지 가설을 같이 세워 줘."
- "뒤쪽 패딩으로 바꾸면 정확도가 어떻게 될지 예측해 볼게 — 실험 설계를 도와줘."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검 — D3 2부작 완결 🎉

**오늘 한 일 3줄**
1. 텍스트→텐서 4단계를 손으로 놓았다 (토큰화 → 사전·정수(train만!) → 임베딩(행 꺼내기 검산) → 앞쪽 패딩)
2. LSTM 분류기(353,538개 — 임베딩 90%)를 D1c 루프로 학습, 야구 vs 우주 ~74%
3. 틀린 문서·내 문장으로 진단하고, "요약 병목"(200단어→h 하나)을 발견했다 — D4의 출발점

**스스로 점검**
- [ ] `<pad>`=0, `<unk>`=1 예약의 이유를 안다
- [ ] 임베딩이 "행 꺼내기"이고 값이 학습됨을 안다
- [ ] 사전을 train으로만 만드는 이유를 M2로 설명할 수 있다
- [ ] 앞쪽 패딩과 마지막 은닉의 관계를 안다

**🔹심화 (선택)**
- **뒤쪽 패딩 실험:** encode를 `ids + [0]*(MAX-len(ids))`로 바꿔 재학습 — 정확도 차이를 확인(앞쪽 패딩의 근거를 실측으로).
- MAX를 50/400으로 바꿔 보세요 — 정보량 vs 패딩 낭비의 트레이드오프.
- `nn.LSTM`을 `nn.GRU`로 교체(반환이 `out, h`로 단순해짐 주의) — 속도·정확도 비교.
- 학습 후 `model.embedding.weight`에서 baseball과 pitcher, baseball과 rocket의 코사인 유사도를 비교해 보세요(`torch.cosine_similarity`) — "비슷한 단어는 비슷한 벡터"의 검증.